# Notebook 2 – ML Workflow, EDA, and Handling Missing Values

In this notebook, we will:

- Revisit the **machine learning workflow**.
- Load a small **tabular dataset** into pandas.
- Perform basic **exploratory data analysis (EDA)**.
- Learn how to **detect** and **handle missing values** in a practical way.

This is the “data understanding and cleaning” phase of our ML journey. Getting this right is often more important than using a fancy model.


## 1. Recap: The Machine Learning Workflow

Remember the big-picture ML workflow:

1. Problem definition
2. Data collection
3. Data understanding and exploration
4. Data cleaning and preprocessing
5. Feature engineering
6. Model training
7. Evaluation and deployment

In this notebook, we are focusing on **steps 3 and 4**:

- Understanding the data: shape, types, distributions.
- Cleaning the data: especially handling **missing values**.

Later notebooks will focus more on feature engineering and model building.


## 2. First look at the dataset

We now have a small synthetic dataset that roughly corresponds to:

- One **row** per student.
- Columns for:
  - Study behavior (`hours_studied`, `extra_classes`)
  - Academic history (`previous_score`)
  - Attendance (`attendance`)
  - Outcome (`final_score`)

We also introduced some **missing values** on purpose so we can practice cleaning.

Let’s explore this dataset step by step.


In [ ]:
import numpy as np
import pandas as pd

# Set a seed for reproducibility (optional)
np.random.seed(42)

# Create a synthetic student-like dataset with some missing values
n = 30  # number of rows

data = {
    "student_id": np.arange(1, n + 1),
    "hours_studied": np.random.randint(0, 10, size=n).astype("float"),
    "previous_score": np.random.randint(30, 90, size=n).astype("float"),
    "attendance": np.random.randint(50, 100, size=n).astype("float"),
    "extra_classes": np.random.choice([0, 1], size=n).astype("float"),
    "final_score": np.random.randint(30, 95, size=n).astype("float"),
}

df = pd.DataFrame(data)

# Introduce some missing values intentionally
for col in ["hours_studied", "previous_score", "attendance"]:
    idx = np.random.choice(df.index, size=3, replace=False)
    df.loc[idx, col] = np.nan

df.head()


### Understanding `shape`, `dtypes`, and `info()`

From the output:

- `shape` tells us how many **rows** and **columns** we have.
- `dtypes` tells us the **data type** of each column (int, float, object/string, etc.).
- `info()` summarizes:
  - Column names
  - Non-null counts (useful for spotting missing values)
  - Data types

If a column shows fewer **non-null** entries than the total number of rows, that means it has **missing values**.


In [ ]:
print("Shape (rows, columns):", df.shape)

print("\nData types:")
print(df.dtypes)

print("\nInfo:")
print(df.info())


### Basic descriptive statistics with `describe()`

`describe()` gives us quick summary statistics for numeric columns:

- **count**: how many non-missing values.
- **mean**, **std**: average and standard deviation.
- **min**, **25%**, **50%**, **75%**, **max**: distribution summary.

We use this to:

- Spot unreasonable values (e.g., negative attendance, impossible scores).
- Get a sense of typical ranges for each feature.


In [3]:
df.describe()


,student_id,hours_studied,previous_score,attendance,extra_classes,final_score
count,30.000000,27.000000,27.000000,27.000000,30.000000,30.000000
mean,15.500000,4.814815,56.962963,75.111111,0.300000,59.400000
std,8.803408,2.689539,19.747408,15.265177,0.466092,19.103349
min,1.000000,0.000000,31.000000,51.000000,0.000000,30.000000
25%,8.250000,3.000000,38.000000,63.500000,0.000000,43.250000
50%,15.500000,5.000000,54.000000,75.000000,0.000000,60.000000
75%,22.750000,7.000000,76.000000,89.000000,1.000000,72.750000
max,30.000000,9.000000,89.000000,99.000000,1.000000,94.000000


## 3. Missing values: what and why?

Missing values are very common in real datasets:

- Students absent on test day ⇒ missing score.
- Sensors failing for a period ⇒ missing readings.
- Users skipping optional fields in forms ⇒ missing attributes.

In pandas, missing numeric values are usually represented as `NaN` (Not a Number).

Basic tools in pandas to detect missing values:

- `df.isna()` or `df.isnull()` → returns a boolean mask.
- `df.isna().sum()` → count missing values per column.
- `df.isna().mean()` → percentage of missing values per column.

We saw above which columns have missing values and roughly how much.


In [4]:
# Overall missing value count
print("Total missing values in each column:")
print(df.isna().sum())

print("\nPercentage of missing values in each column:")
print(df.isna().mean() * 100)

Total missing values in each column:
student_id        0
hours_studied     3
previous_score    3
attendance        3
extra_classes     0
final_score       0
dtype: int64

Percentage of missing values in each column:
student_id         0.0
hours_studied     10.0
previous_score    10.0
attendance        10.0
extra_classes      0.0
final_score        0.0
dtype: float64


## 4. Strategies for handling missing values

There is no single “correct” way to handle missing values. Common strategies include:

1. **Drop rows** with missing values.
   - Simple, but you lose data.
   - Okay when:
     - Only a small fraction of rows are affected.
     - Rows are missing values randomly.

2. **Drop columns** with too many missing values.
   - When a column is mostly missing and not very informative.

3. **Impute (fill) missing values** with:
   - A constant (0, -1, "Unknown", etc.).
   - Mean / median / mode of that column.
   - More advanced methods (knn, regression, model-based imputation).

4. **Leave them for the model** (for some algorithms that can handle NaNs), but many ML models break if you don’t handle them first.

In this notebook, we will practice:

- Dropping rows with missing values.
- Filling missing values with simple statistics (mean/median/mode).


In [5]:
# How many rows have at least one missing value?
rows_with_any_missing = df.isna().any(axis=1).sum()
print(f"Rows with at least one missing value: {rows_with_any_missing} out of {len(df)}")

# Show a few rows that contain missing values
df[df.isna().any(axis=1)].head()


Rows with at least one missing value: 9 out of 30


,student_id,hours_studied,previous_score,attendance,extra_classes,final_score
4,5,NaN,36.0,51.0,0.0,63.0
9,10,4.0,33.0,NaN,0.0,66.0
12,13,7.0,43.0,NaN,0.0,94.0
15,16,4.0,38.0,NaN,1.0,30.0
16,17,NaN,55.0,80.0,1.0,34.0


### Dropping rows: pros and cons

Dropping rows with missing values is the **simplest** approach:

- Pros:
  - Easy to implement (`dropna()`).
  - Keeps the remaining data “clean”.

- Cons:
  - You may lose a lot of data if missingness is common.
  - If the missingness is **not random**, you might introduce bias.

In real projects, dropping rows is usually only acceptable when:

- The dataset is large.
- The percentage of affected rows is small.


In [6]:
# Make a copy so we don't lose the original
df_drop_rows = df.copy()

print("Shape before dropping:", df_drop_rows.shape)
df_drop_rows = df_drop_rows.dropna()
print("Shape after dropping rows with any missing values:", df_drop_rows.shape)

df_drop_rows.head()


Shape before dropping: (30, 6)
Shape after dropping rows with any missing values: (21, 6)


,student_id,hours_studied,previous_score,attendance,extra_classes,final_score
0,1,6.0,86.0,85.0,0.0,70.0
1,2,3.0,32.0,99.0,1.0,57.0
2,3,7.0,66.0,89.0,1.0,36.0
3,4,4.0,80.0,53.0,1.0,41.0
5,6,9.0,50.0,55.0,0.0,62.0


### Filling missing values with mean/median

Simple imputation strategy:

- For numeric columns:
  - Replace missing values with the **mean** or **median** of that column.
- For categorical columns:
  - Replace missing values with the **mode** (most frequent value).

Advantages:

- Keeps all rows.
- Easy to implement and fast.

Disadvantages:

- Can slightly distort distributions.
- Ignores relationships between columns.
- Not always the best statistical choice, but good enough for many beginner projects.

For this bootcamp, mean/median/mode imputation is usually a good default starting point.


In [7]:
# Another copy to experiment with imputation
df_impute = df.copy()

print("Missing values before imputation:")
print(df_impute.isna().sum())

# For numeric columns, we can fill missing with mean or median.
num_cols = ["hours_studied", "previous_score", "attendance"]

for col in num_cols:
    col_mean = df_impute[col].mean()
    df_impute[col] = df_impute[col].fillna(col_mean)

print("\nMissing values after mean imputation:")
print(df_impute.isna().sum())


Missing values before imputation:
student_id        0
hours_studied     3
previous_score    3
attendance        3
extra_classes     0
final_score       0
dtype: int64

Missing values after mean imputation:
student_id        0
hours_studied     0
previous_score    0
attendance        0
extra_classes     0
final_score       0
dtype: int64


In [8]:
print("Original data (first 5 rows, may contain NaNs):")
print(df.head())

print("\nAfter mean imputation (first 5 rows):")
print(df_impute.head())


Original data (first 5 rows, may contain NaNs):
   student_id  hours_studied  previous_score  attendance  extra_classes  \
0           1            6.0            86.0        85.0            0.0   
1           2            3.0            32.0        99.0            1.0   
2           3            7.0            66.0        89.0            1.0   
3           4            4.0            80.0        53.0            1.0   
4           5            NaN            36.0        51.0            0.0   

   final_score  
0         70.0  
1         57.0  
2         36.0  
3         41.0  
4         63.0  

After mean imputation (first 5 rows):
   student_id  hours_studied  previous_score  attendance  extra_classes  \
0           1       6.000000            86.0        85.0            0.0   
1           2       3.000000            32.0        99.0            1.0   
2           3       7.000000            66.0        89.0            1.0   
3           4       4.000000            80.0        53.0    

In [9]:
# Choose which cleaned version to carry forward
# Here we use the imputed version
clean_df = df_impute.copy()

# Optionally, save to CSV so Notebook 3 can load it
clean_df.to_csv("students_clean.csv", index=False)

clean_df.head()


,student_id,hours_studied,previous_score,attendance,extra_classes,final_score
0,1,6.000000,86.0,85.0,0.0,70.0
1,2,3.000000,32.0,99.0,1.0,57.0
2,3,7.000000,66.0,89.0,1.0,36.0
3,4,4.000000,80.0,53.0,1.0,41.0
4,5,4.814815,36.0,51.0,0.0,63.0


## 5. Wrap up: what we achieved in this notebook

In this notebook, you:

- Loaded a small **tabular dataset** into pandas.
- Used `shape`, `dtypes`, `info()`, and `describe()` to understand the data.
- Detected **missing values** using `isna()` and simple counts.
- Practiced two basic strategies for handling missing values:
  - Dropping rows with missing data.
  - Filling (imputing) missing values with the column mean.

In real projects, data cleaning can take **more time than model training**.
But the payoff is huge: cleaner data → more reliable models.

In the next notebook, we’ll take this cleaned dataset and:

- Split it into train/test sets.
- Build our first **end-to-end ML model** on tabular data.
- Start connecting this to TensorFlow/Keras models.
